In [15]:
# ============================================
# 1. Configurações iniciais - Silver to Gold
# ============================================

import pandas as pd
import numpy as np
from pathlib import Path
import gdown

# Caminhos base (ajuste se necessário)
PROJECT_ROOT = Path("/content/drive/MyDrive/PROJETOS/FINANCEIRO/ETL/DATA")
SILVER_PATH = Path('/content/SILVER')
GOLD_PATH = PROJECT_ROOT / "GOLD"

GOLD_PATH.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SILVER_PATH :", SILVER_PATH)
print("GOLD_PATH   :", GOLD_PATH)


PROJECT_ROOT: /content/drive/MyDrive/PROJETOS/FINANCEIRO/ETL/DATA
SILVER_PATH : /content/SILVER
GOLD_PATH   : /content/drive/MyDrive/PROJETOS/FINANCEIRO/ETL/DATA/GOLD


In [16]:
URL = "https://drive.google.com/drive/folders/1hCfTUJLkvJOnsd_JftKvQzeYBCIi3Fit?usp=sharing"

arquivos_silver = gdown.download_folder(URL)

Retrieving folder contents


Processing file 1im87xcTZ5K1tWPPurBQzqQiIJ-5RKZED calendario_silver.parquet
Processing file 1scyMs-HssM2Zy8qW2vmSZyHzK4gHrlM2 centros_custo_silver.parquet
Processing file 1dIl9c2kAclKknpTvhyCamvuWvwPb-rx_ clientes_silver.parquet
Processing file 17KLJTzdbz2Bn_mzsPbhu5zOdzkN6rDf7 contas_pagar_silver.parquet
Processing file 1uhDMVBTRNVpnziX_4bjiEeCya-ydcp26 contas_receber_silver.parquet
Processing file 1ZxnnXexByIeLQ6NmLssD7w8koQN1fvAi fornecedores_silver.parquet
Processing file 1cTtsmTiieLZd4dv5vWSaWYpYfZtNfRO0 plano_contas_silver.parquet


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1im87xcTZ5K1tWPPurBQzqQiIJ-5RKZED
To: /content/SILVER/calendario_silver.parquet
100%|██████████| 15.2k/15.2k [00:00<00:00, 29.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1scyMs-HssM2Zy8qW2vmSZyHzK4gHrlM2
To: /content/SILVER/centros_custo_silver.parquet
100%|██████████| 2.58k/2.58k [00:00<00:00, 6.57MB/s]
Downloading...
From: https://drive.google.com/uc?id=1dIl9c2kAclKknpTvhyCamvuWvwPb-rx_
To: /content/SILVER/clientes_silver.parquet
100%|██████████| 4.16k/4.16k [00:00<00:00, 11.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=17KLJTzdbz2Bn_mzsPbhu5zOdzkN6rDf7
To: /content/SILVER/contas_pagar_silver.parquet
100%|██████████| 27.7k/27.7k [00:00<00:00, 20.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1uhDMVBTRNVpnziX_4bjiEeCya-ydcp26
To: /content/SILVER/contas_receber_silver.parquet
100%|██████████|

In [17]:
# ============================================
# 2. Leitura dos dados Silver
# ============================================

# Dimensões Silver
df_clientes = pd.read_parquet(SILVER_PATH / "clientes_silver.parquet")
df_fornecedores = pd.read_parquet(SILVER_PATH / "fornecedores_silver.parquet")
df_centros_custo = pd.read_parquet(SILVER_PATH / "centros_custo_silver.parquet")
df_plano_contas = pd.read_parquet(SILVER_PATH / "plano_contas_silver.parquet")
df_calendario = pd.read_parquet(SILVER_PATH / "calendario_silver.parquet")

# Fatos Silver
df_cr = pd.read_parquet(SILVER_PATH / "contas_receber_silver.parquet")
df_cp = pd.read_parquet(SILVER_PATH / "contas_pagar_silver.parquet")

print("clientes_silver        :", df_clientes.shape)
print("fornecedores_silver    :", df_fornecedores.shape)
print("centros_custo_silver   :", df_centros_custo.shape)
print("plano_contas_silver    :", df_plano_contas.shape)
print("calendario_silver      :", df_calendario.shape)
print("contas_receber_silver  :", df_cr.shape)
print("contas_pagar_silver    :", df_cp.shape)


clientes_silver        : (50, 5)
fornecedores_silver    : (40, 5)
centros_custo_silver   : (8, 3)
plano_contas_silver    : (10, 5)
calendario_silver      : (731, 7)
contas_receber_silver  : (400, 16)
contas_pagar_silver    : (350, 16)


In [18]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [19]:
# ============================================
# 3. Construção das dimensões Gold
# ============================================

# --------------------------
# 3.1 dim_tempo
# --------------------------

# Seleciona colunas relevantes do calendário
dim_tempo = df_calendario[[
    "id_tempo",
    "data",
    "ano",
    "mes",
    "nome_mes",
    "dia",
    "nome_dia_semana"
]].copy()

# Garante unicidade da chave id_tempo
dim_tempo = dim_tempo.drop_duplicates(subset=["id_tempo"]).sort_values("data").reset_index(drop=True)

# (Opcional) criar ano_mes e trimestre para facilitar análises no BI
dim_tempo["ano_mes"] = dim_tempo["data"].dt.to_period("M").astype(str)
dim_tempo["trimestre"] = dim_tempo["data"].dt.to_period("Q").astype(str)

print("dim_tempo:")
print(dim_tempo.head())

# --------------------------
# 3.2 dim_cliente
# --------------------------

dim_cliente = df_clientes[[
    "id_cliente",
    "nome_cliente",
    "segmento",
    "cidade",
    "estado"
]].copy()

# Garante unicidade de id_cliente
dim_cliente = dim_cliente.drop_duplicates(subset=["id_cliente"]).reset_index(drop=True)

# (Opcional) criar região a partir do estado
regioes = {
    "SP": "Sudeste", "RJ": "Sudeste", "MG": "Sudeste",
    "PR": "Sul", "SC": "Sul", "RS": "Sul",
    "BA": "Nordeste", "PE": "Nordeste"
}
dim_cliente["regiao"] = dim_cliente["estado"].map(regioes).fillna("Outros")

print("\ndim_cliente:")
print(dim_cliente.head())

# --------------------------
# 3.3 dim_fornecedor
# --------------------------

dim_fornecedor = df_fornecedores[[
    "id_fornecedor",
    "nome_fornecedor",
    "tipo_fornecedor",
    "cidade",
    "estado"
]].copy()

dim_fornecedor = dim_fornecedor.drop_duplicates(subset=["id_fornecedor"]).reset_index(drop=True)

print("\ndim_fornecedor:")
print(dim_fornecedor.head())

# --------------------------
# 3.4 dim_centro_custo
# --------------------------

dim_centro_custo = df_centros_custo[[
    "id_centro_custo",
    "nome_centro_custo",
    "tipo"
]].copy()

dim_centro_custo = dim_centro_custo.drop_duplicates(subset=["id_centro_custo"]).reset_index(drop=True)

print("\ndim_centro_custo:")
print(dim_centro_custo.head())

# --------------------------
# 3.5 dim_plano_contas
# --------------------------

dim_plano_contas = df_plano_contas[[
    "id_plano_contas",
    "codigo_contabil",
    "descricao",
    "grupo",
    "tipo"
]].copy()

dim_plano_contas = dim_plano_contas.drop_duplicates(subset=["id_plano_contas"]).reset_index(drop=True)

print("\ndim_plano_contas:")
print(dim_plano_contas.head())


dim_tempo:
   id_tempo       data   ano  mes nome_mes  dia nome_dia_semana  ano_mes  \
0  20240101 2024-01-01  2024    1  January    1          Monday  2024-01   
1  20240102 2024-01-02  2024    1  January    2         Tuesday  2024-01   
2  20240103 2024-01-03  2024    1  January    3       Wednesday  2024-01   
3  20240104 2024-01-04  2024    1  January    4        Thursday  2024-01   
4  20240105 2024-01-05  2024    1  January    5          Friday  2024-01   

  trimestre  
0    2024Q1  
1    2024Q1  
2    2024Q1  
3    2024Q1  
4    2024Q1  

dim_cliente:
   id_cliente nome_cliente    segmento         cidade estado    regiao
0           1  Cliente 001  Tecnologia       Londrina     PR       Sul
1           2  Cliente 002  Tecnologia      Joinville     SC       Sul
2           3  Cliente 003   Indústria       Curitiba     PR       Sul
3           4  Cliente 004   Indústria         Recife     PE  Nordeste
4           5  Cliente 005    Comércio  Caxias do Sul     RS       Sul

dim_for

In [20]:
# ============================================
# 4. Dicionário de tempo para mapear datas
# ============================================

# Cria um dicionário data -> id_tempo para reutilizar nas fatos
dict_data_to_id_tempo = dim_tempo.set_index("data")["id_tempo"].to_dict()

print("Exemplos de mapeamento data -> id_tempo (primeiros 5):")
for i, (k, v) in enumerate(dict_data_to_id_tempo.items()):
    print(k, "->", v)
    if i >= 4:
        break


Exemplos de mapeamento data -> id_tempo (primeiros 5):
2024-01-01 00:00:00 -> 20240101
2024-01-02 00:00:00 -> 20240102
2024-01-03 00:00:00 -> 20240103
2024-01-04 00:00:00 -> 20240104
2024-01-05 00:00:00 -> 20240105


In [21]:
# ============================================
# 5. Fato CONTAS A RECEBER (Gold)
# ============================================

fato_cr = df_cr.copy()

# Cria chaves de tempo (emissão, vencimento, pagamento)
fato_cr["id_tempo_emissao"] = fato_cr["data_emissao"].map(dict_data_to_id_tempo).astype("Int64")
fato_cr["id_tempo_vencimento"] = fato_cr["data_vencimento"].map(dict_data_to_id_tempo).astype("Int64")
fato_cr["id_tempo_pagamento"] = fato_cr["data_pagamento"].map(dict_data_to_id_tempo).astype("Int64")

# Define colunas finais da fato (modelo estrela)
cols_fato_cr = [
    "id_conta_receber",
    "id_cliente",
    "id_centro_custo",
    "id_plano_contas",
    "id_tempo_emissao",
    "id_tempo_vencimento",
    "id_tempo_pagamento",
    # Medidas
    "valor_original",
    "valor_pago",
    "juros_multa",
    "desconto",
    "valor_em_aberto",
    "dias_atraso",
    # Flags e atributos de apoio
    "status",
    "flag_atrasada",
    "flag_em_aberto",
    # Datas brutas (úteis para debug, opcionais no BI)
    "data_emissao",
    "data_vencimento",
    "data_pagamento"
]

fato_contas_receber = fato_cr[cols_fato_cr].copy()

print("fato_contas_receber:")
print(fato_contas_receber.head())
print("\nDtypes:")
print(fato_contas_receber.dtypes)

print("\nResumo valor_em_aberto (receber):")
print(fato_contas_receber["valor_em_aberto"].describe())


fato_contas_receber:
   id_conta_receber  id_cliente  id_centro_custo  id_plano_contas  \
0                 1           5                1                1   
1                 2          27                7                2   
2                 3          26                4                2   
3                 4          23                5                3   
4                 5          18                8                2   

   id_tempo_emissao  id_tempo_vencimento  id_tempo_pagamento  valor_original  \
0          20241116             20241223                <NA>        14098.68   
1          20250709             20250823                <NA>         2998.22   
2          20251107             20251130                <NA>        16543.85   
3          20250208             20250226            20250228        16638.81   
4          20251209                 <NA>                <NA>         4295.45   

   valor_pago  juros_multa  desconto  valor_em_aberto  dias_atraso     status  \
0 

In [22]:
# ============================================
# 6. Fato CONTAS A PAGAR (Gold)
# ============================================

fato_cp = df_cp.copy()

# Cria chaves de tempo (emissão, vencimento, pagamento)
fato_cp["id_tempo_emissao"] = fato_cp["data_emissao"].map(dict_data_to_id_tempo).astype("Int64")
fato_cp["id_tempo_vencimento"] = fato_cp["data_vencimento"].map(dict_data_to_id_tempo).astype("Int64")
fato_cp["id_tempo_pagamento"] = fato_cp["data_pagamento"].map(dict_data_to_id_tempo).astype("Int64")

# Define colunas finais da fato (modelo estrela)
cols_fato_cp = [
    "id_conta_pagar",
    "id_fornecedor",
    "id_centro_custo",
    "id_plano_contas",
    "id_tempo_emissao",
    "id_tempo_vencimento",
    "id_tempo_pagamento",
    # Medidas
    "valor_original",
    "valor_pago",
    "juros_multa",
    "desconto",
    "valor_em_aberto",
    "dias_atraso",
    # Flags e atributos de apoio
    "status",
    "flag_atrasada",
    "flag_em_aberto",
    # Datas brutas (opcional)
    "data_emissao",
    "data_vencimento",
    "data_pagamento"
]

fato_contas_pagar = fato_cp[cols_fato_cp].copy()

print("fato_contas_pagar:")
print(fato_contas_pagar.head())
print("\nDtypes:")
print(fato_contas_pagar.dtypes)

print("\nResumo valor_em_aberto (pagar):")
print(fato_contas_pagar["valor_em_aberto"].describe())


fato_contas_pagar:
   id_conta_pagar  id_fornecedor  id_centro_custo  id_plano_contas  \
0               1             28                4               12   
1               2             28                7               14   
2               3              6                1               10   
3               4             12                1               15   
4               5             24                8               11   

   id_tempo_emissao  id_tempo_vencimento  id_tempo_pagamento  valor_original  \
0          20240711             20240806            20240817         7965.22   
1          20240702             20240714            20240807         2504.14   
2          20250521             20250620            20250715         2513.83   
3          20251221                 <NA>            20251231        11296.64   
4          20250705             20250731            20250809         9049.28   

   valor_pago  juros_multa  desconto  valor_em_aberto  dias_atraso status  \
0 

In [23]:
# ============================================
# 7. Checks de integridade referencial
# ============================================

# ---- Fato receber x dim_cliente
mask_fk_cli_missing = ~fato_contas_receber["id_cliente"].isin(dim_cliente["id_cliente"])
print("Linhas em fato_contas_receber com id_cliente inexistente em dim_cliente:", mask_fk_cli_missing.sum())

# ---- Fato pagar x dim_fornecedor
mask_fk_forn_missing = ~fato_contas_pagar["id_fornecedor"].isin(dim_fornecedor["id_fornecedor"])
print("Linhas em fato_contas_pagar com id_fornecedor inexistente em dim_fornecedor:", mask_fk_forn_missing.sum())

# ---- Fato receber/pagar x dim_plano_contas
mask_fk_pc_cr_missing = ~fato_contas_receber["id_plano_contas"].isin(dim_plano_contas["id_plano_contas"])
mask_fk_pc_cp_missing = ~fato_contas_pagar["id_plano_contas"].isin(dim_plano_contas["id_plano_contas"])

print("Linhas em fato_contas_receber com id_plano_contas inexistente:", mask_fk_pc_cr_missing.sum())
print("Linhas em fato_contas_pagar com id_plano_contas inexistente:", mask_fk_pc_cp_missing.sum())

# ---- Fato receber/pagar x dim_centro_custo (desconsiderando nulos)
mask_cc_cr_notna = fato_contas_receber["id_centro_custo"].notna()
mask_cc_cp_notna = fato_contas_pagar["id_centro_custo"].notna()

mask_fk_cc_cr_missing = mask_cc_cr_notna & ~fato_contas_receber["id_centro_custo"].isin(dim_centro_custo["id_centro_custo"])
mask_fk_cc_cp_missing = mask_cc_cp_notna & ~fato_contas_pagar["id_centro_custo"].isin(dim_centro_custo["id_centro_custo"])

print("Linhas em fato_contas_receber com id_centro_custo inexistente:", mask_fk_cc_cr_missing.sum())
print("Linhas em fato_contas_pagar com id_centro_custo inexistente:", mask_fk_cc_cp_missing.sum())

# ---- Fato receber/pagar x dim_tempo (emissão/vencimento)
for col in ["id_tempo_emissao", "id_tempo_vencimento"]:
    mask_time_cr_missing = ~fato_contas_receber[col].isin(dim_tempo["id_tempo"])
    mask_time_cp_missing = ~fato_contas_pagar[col].isin(dim_tempo["id_tempo"])
    print(f"Linhas em fato_contas_receber com {col} inexistente em dim_tempo:", mask_time_cr_missing.sum())
    print(f"Linhas em fato_contas_pagar com {col} inexistente em dim_tempo:", mask_time_cp_missing.sum())


Linhas em fato_contas_receber com id_cliente inexistente em dim_cliente: 0
Linhas em fato_contas_pagar com id_fornecedor inexistente em dim_fornecedor: 0
Linhas em fato_contas_receber com id_plano_contas inexistente: 0
Linhas em fato_contas_pagar com id_plano_contas inexistente: 0
Linhas em fato_contas_receber com id_centro_custo inexistente: 0
Linhas em fato_contas_pagar com id_centro_custo inexistente: 0
Linhas em fato_contas_receber com id_tempo_emissao inexistente em dim_tempo: 0
Linhas em fato_contas_pagar com id_tempo_emissao inexistente em dim_tempo: 0
Linhas em fato_contas_receber com id_tempo_vencimento inexistente em dim_tempo: 11
Linhas em fato_contas_pagar com id_tempo_vencimento inexistente em dim_tempo: 10


In [24]:
# ============================================
# 8. Salvando a camada GOLD
# ============================================

# Dimensões
dim_tempo.to_parquet(GOLD_PATH / "dim_tempo.parquet", index=False)
dim_cliente.to_parquet(GOLD_PATH / "dim_cliente.parquet", index=False)
dim_fornecedor.to_parquet(GOLD_PATH / "dim_fornecedor.parquet", index=False)
dim_centro_custo.to_parquet(GOLD_PATH / "dim_centro_custo.parquet", index=False)
dim_plano_contas.to_parquet(GOLD_PATH / "dim_plano_contas.parquet", index=False)

# Fatos
fato_contas_receber.to_parquet(GOLD_PATH / "fato_contas_receber.parquet", index=False)
fato_contas_pagar.to_parquet(GOLD_PATH / "fato_contas_pagar.parquet", index=False)

print("Arquivos GOLD salvos em:", GOLD_PATH)


Arquivos GOLD salvos em: /content/drive/MyDrive/PROJETOS/FINANCEIRO/ETL/DATA/GOLD
